# Question Answering with LangChain, OpenAI, and MultiQuery Retriever


<br>

## Intro

In this lab you'll build a Retrieval-Augmented Generation (RAG) chatbot backed by Elasticsearch and OpenAI, using LangChain's `MultiQueryRetriever` to improve retrieval quality.

**Why use a `MultiQueryRetriever`?** A single user question, phrased one way, might miss documents that use different wording for the same idea. [`MultiQueryRetriever`](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) asks an LLM to rephrase the original question into several alternative versions, runs a separate search for each version, and merges the results — giving you a broader, more relevant set of documents than a single search would.

Here's how the workflow will look like:

- **Setup (once):** split a sample dataset of documents into passages (chunks), turn each passage into an embedding, and store them in a vector database (in this case, we'll use Elasticsearch).
- **At question time:** the `MultiQueryRetriever` generates a few variations of the question, searches Elasticsearch with each one, and combines the retrieved passages. Those passages are then passed to an LLM, which uses them as context to write the final answer.


And here's what to expect from this lab:

1. **Complete a guided demo** — You'll be given the overall structure and process to follow: connecting to Elasticsearch and OpenAI, loading and chunking a sample set of workplace documents, embedding and indexing them into a vector store, and wiring up a `MultiQueryRetriever` chain that expands a question into several variations before retrieving and answering. Some steps are already implemented for you, and you'll need to complete the rest (e.g. filling in missing parameters and logic) to get it fully working.
2. **Replicate it yourself with variations** — Then, you'll create **at least two new iterations** of the question-answering step (new questions and/or settings) to explore how `MultiQueryRetriever` behaves.

By the end of this lab, you'll understand how multi-query retrieval works and be able to apply it to your own RAG pipelines.

<br>

## Install and import dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [1]:
!pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2" "langchain-elasticsearch<0.3" "jq==1.12.0"

     ---------------------------------------- 0.0/71.9 kB ? eta -:--:--
     ----------------- ---------------------- 30.7/71.9 kB 1.3 MB/s eta 0:00:01
     ---------------------------------------- 71.9/71.9 kB 1.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/412.1 kB ? eta -:--:--
   --------------------------- ------------ 286.7/412.1 kB 5.9 MB/s eta 0:00:01
   ---------------------------------------- 412.1/412.1 kB 5.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/952.8 kB ? eta -:--:--
   --------- ------------------------------ 225.3/952.8 kB 6.7 MB/s eta 0:00:01
   ----------------- ---------------------- 409.6/952.8 kB 8.5 MB/s eta 0:00:01
   ----------------------------- ---------- 696.3/952.8 kB 5.5 MB/s eta 0:00:01
   ---------------------------------------- 952.8/952.8 kB 6.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/65.1 kB ? eta -:--:--
   ---------------------------------------- 65.1/65.1 kB ? eta 0:00:00
   --


[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain_openai import ChatOpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

## Connect to Elasticsearch

In this lab, you'll use Elasticsearch as the vector database and search engine that stores your documents and retrieves the most relevant ones in response to user queries.

**What is Elasticsearch?**

Elasticsearch is a distributed search and analytics engine designed for fast retrieval over large collections of data. In AI applications, it can store both text and vector embeddings, making it well suited for Retrieval-Augmented Generation (RAG) workflows where an LLM retrieves relevant context before generating a response.


**Getting started with Elastic Cloud:**

1. If you don't already have one, sign up for a free trial: [https://www.elastic.co/](https://www.elastic.co/).
2. During sign-up (or afterwards), create a deployment/project. Either a classic deployment or a serverless project will work for this lab.
3. Once your deployment/project is ready, you'll need to find your **Cloud ID** and create an **API key**. 

<br>

> To find your **Cloud ID** and create an **API key**, 
> follow the instructions on the link below.
> 
>   👇👇👇
>
> https://www.elastic.co/search-labs/tutorials/install-elasticsearch/find-cloud-id-create-api-keys 📌
>
<br>

In [4]:
#
#
# To find your Cloud ID and create an API key, 
# follow the instructions on the link below:
#
# 👇👇👇
#
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/find-cloud-id-create-api-keys 📌
#
#
ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name=None, #give it a meaningful name
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [13]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [15]:
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    # Populate the five requested metadata fields for the document prompt.
    for key in ("name", "summary", "url", "category", "updated_at"):
        metadata[key] = record.get(key) or "Unknown"

    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800, chunk_overlap=400 # 800 tokens, 400-token overlap
)
docs = loader.load_and_split(text_splitter=text_splitter)

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [18]:
# Fix invalid date metadata
for doc in docs:
    updated_at = doc.metadata.get("updated_at")

    if updated_at in (None, "", "Unknown", "Not available"):
        doc.metadata.pop("updated_at", None)

print("Invalid date metadata cleaned!")

Invalid date metadata cleaned!


In [21]:
documents = ElasticsearchStore.from_documents(
    documents=docs,
    embedding=embeddings,
    index_name="multi-query-chatbot-lab",
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)
# Use the populated index for retrieval.
vectorstore = documents

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [22]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['What does the NASA sales team do and what are their responsibilities?  ', 'Can you provide information about the structure and functions of the NASA sales team?  ', 'Who are the members of the NASA sales team and what roles do they play?']


---- Answer ----
content='The NASA sales team refers to the sales organization that operates within the North America and South America region, which is abbreviated as NASA. This region encompasses the United States, Canada, Mexico, as well as Central and South America. The NASA sales team is structured to effectively serve customers in these areas and achieve business objectives.\n\nThe leadership of the NASA sales team is divided between two Area Vice-Presidents: Laura Martinez, who oversees North America, and Gary Johnson, who is responsible for South America. Each regional sales team, including the NASA team, consists of dedicated account managers, sales representatives, and support staff. Their primary responsibilities include identifying and pursuing new business opportunities, nurturing existing client relationships, and ensuring customer satisfaction.\n\nThe NASA sales team collaborates closely with other departments, such as marketing, product development, and customer support

<br>

## 🚀 Your turn

Time to make this lab your own!


So far we've asked a single question ("what is the nasa sales team?") and looked at one answer. Now it's your turn to try the pipeline with your own questions and see how `MultiQueryRetriever` handles them.


Here's what to do:

1. **Pick 2 new questions** — Look through the sample dataset (or just get creative) and write down 2 questions you'd like the chatbot to answer, different from the example above.
2. **Ask each question** — For each question, copy the last code cell, replace the question passed to `chain.invoke(...)`, and run it.
3. **Check the logs** — Because logging is enabled, you'll see the alternative questions `MultiQueryRetriever` generated for you in the cell output. Take a look at them.
4. **Reflect** — In a markdown cell, briefly note: Did the generated questions look useful? Was the final answer correct and well grounded in the retrieved context, or did the model seem to be missing information?

<br>

💡 **Tip:**

- If an answer looks off, try asking the same question in a different way, or peek at the generated queries to see if they drifted from what you actually meant.

<br>

⭐️ **Bonus ideas:**

- Change `retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)` to instead use `vectorstore.as_retriever()` directly (no multi-query), and compare the answers you get for the same questions. Does `MultiQueryRetriever` actually help?
- Look at the `k` parameter of `as_retriever()` (e.g. `vectorstore.as_retriever(search_kwargs={"k": 3})`) to control how many passages are retrieved per query, and see how that affects the final answer.
- Edit `LLM_CONTEXT_PROMPT` to change the tone or format of the final answer (e.g. ask for a short bullet-point summary instead of a verbose answer), and see how that changes the output.
- Ask a question that isn't covered by the sample dataset at all — does the chatbot correctly say it doesn't know, or does it make something up?

In [23]:
# Exercise: ask two new questions using the existing MultiQueryRetriever pipeline.
questions = [
    "Which people or teams are mentioned in the documents, and what do they do?",
    "What information do the documents provide about products or services?",
]

for number, question in enumerate(questions, start=1):
    print(f"\n--- Question {number}: {question} ---")
    answer = chain.invoke(question)
    print(answer)
    print("Check the MultiQueryRetriever INFO logs above for generated queries.")


--- Question 1: Which people or teams are mentioned in the documents, and what do they do? ---


INFO:langchain.retrievers.multi_query:Generated queries: ['What individuals or groups are referenced in the documents, and what are their roles or activities?', 'Can you identify the people or teams discussed in the documents and describe their functions or contributions?', 'Who are the key figures or teams highlighted in the documents, and what actions or responsibilities do they have?']


content="The documents mention several key groups and individuals within the company, each with distinct roles and responsibilities:\n\n1. **New Employees**: The onboarding guide is primarily directed at new employees, providing them with essential information about company policies, procedures, and resources. New employees are encouraged to familiarize themselves with the company culture, complete required training, and update their tax information and benefits enrollment.\n\n2. **Human Resources (HR) Department**: The HR department plays a crucial role in the onboarding process, assisting new employees with questions regarding policies, benefits, and tax documentation. They are responsible for ensuring that employees complete necessary forms and training, and they provide support for any workplace accommodations.\n\n3. **Sales Team**: The sales team is responsible for promoting and selling the company's products and services. They establish relationships with customers, understand th

INFO:langchain.retrievers.multi_query:Generated queries: ['What details do the documents contain regarding the features or offerings of products and services?  ', 'How do the documents describe the characteristics or benefits of various products and services?  ', 'What insights can be gathered from the documents about the available products or services?']


content="The documents provide several key insights regarding the products and services of the tech company for fiscal year 2024. Here are the main points:\n\n1. **Product/Service Portfolio Optimization**: The company plans to optimize its existing product and service offerings by focusing on high-demand solutions. This indicates a strategic approach to ensure that the products and services being offered align with market needs and customer preferences.\n\n2. **New Product Launches**: The strategy outlines an objective to launch at least two new products or services in high-demand market segments. This reflects the company's commitment to innovation and responsiveness to market trends, aiming to capture new opportunities and meet evolving customer needs.\n\n3. **Emerging Technology Areas**: There is an emphasis on developing and launching innovative products and services in emerging technology areas. This suggests that the company is not only looking to enhance its current offerings bu

In [24]:
# Optional bonus: compare ordinary single-query retrieval with MultiQueryRetriever.
# Uses the same LLM and context prompt so only the retriever changes.
single_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

single_context = RunnableParallel(
    context=single_retriever | _combine_documents,
    question=RunnablePassthrough(),
)
single_query_chain = single_context | LLM_CONTEXT_PROMPT | llm

question = questions[0]
print("Question:", question)
print("\nSingle-query answer:\n", single_query_chain.invoke(question))
print("\nMulti-query answer:\n", chain.invoke(question))

Question: Which people or teams are mentioned in the documents, and what do they do?

Single-query answer:
 content="The documents provided mention several groups of people and teams, primarily focusing on new employees and IT teams within the company. Here’s a detailed breakdown of the mentioned groups and their roles:\n\n1. **New Employees**: The onboarding guide is specifically tailored for new employees who have recently joined the company. It outlines essential steps for their integration into the organization, including attending orientation, reviewing company policies, completing required training, updating tax information, enrolling in benefits, and settling into their workspace. The guide emphasizes the importance of these steps to ensure a smooth transition and to familiarize new hires with the company culture and values.\n\n2. **HR Department**: The Human Resources (HR) department plays a crucial role in supporting new employees throughout the onboarding process. They are re

INFO:langchain.retrievers.multi_query:Generated queries: ['What individuals or groups are referenced in the documents, and what are their roles or activities?', 'Can you identify the people or teams discussed in the documents and describe their functions or contributions?', 'Who are the key figures or teams highlighted in the documents, and what actions or responsibilities do they have?']



Multi-query answer:
 content="The documents mention several key groups and individuals within the company, each with distinct roles and responsibilities:\n\n1. **New Employees**: The onboarding guide is primarily directed at new employees, providing them with essential information about company policies, procedures, and resources. New employees are encouraged to familiarize themselves with the company culture, complete required training, and update their tax information and benefits enrollment.\n\n2. **Human Resources (HR) Department**: The HR department plays a crucial role in the onboarding process, assisting new employees with questions, managing tax elections and documents, and providing information about benefits. They are also responsible for ensuring that employees understand and comply with company policies, including the Intellectual Property Policy.\n\n3. **Sales Team**: The sales team is responsible for promoting and selling the company's products and services. They establi